In [50]:
import config
import importlib
importlib.reload(config) # tải lại module lấy phiên bản mới nhất

print("RANDOM_SEED:", config.RANDOM_SEED)
print("RAW_DATA_PATH:", config.RAW_DATA_PATH)

RANDOM_SEED: 42
RAW_DATA_PATH: data/raw/hour.csv


## 1. Library Import

In [51]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score
from sklearn.feature_selection import mutual_info_classif
from sklearn.inspection import permutation_importance

pd.set_option('display.max_columns', None) #setup hiển thị toàn bộ các cột của DF
seed = config.RANDOM_SEED #gọi seed từ file config

## 2. Data Load

In [52]:
df = pd.read_csv(config.RAW_DATA_PATH)
df.head()

,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,3,13,16
1,2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,8,32,40
2,3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,5,27,32
3,4,2011-01-01,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,3,10,13
4,5,2011-01-01,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0,0,1,1


In [53]:
df['dteday'] = pd.to_datetime(df['dteday'])
df.dtypes

instant                int64
dteday        datetime64[ns]
season                 int64
yr                     int64
mnth                   int64
hr                     int64
holiday                int64
weekday                int64
workingday             int64
weathersit             int64
temp                 float64
atemp                float64
hum                  float64
windspeed            float64
casual                 int64
registered             int64
cnt                    int64
dtype: object

## 3. Feature Encode

tạo 2 biến:

'lag_1h': độ chênh lệch số lượt thuê xe trong 1h

'same_hour_prev': số lượt thuê xe cùng giờ ngày hôm qua

In [54]:
df["time_full"] = df["dteday"] + pd.to_timedelta(df["hr"], unit="h")
df["gap_hours"] = df["time_full"].diff().dt.total_seconds() / 3600

df["lag_1h"] = df["cnt"].shift(1)
df.loc[df["gap_hours"] != 1, "lag_1h"] = np.nan

same_hour_prev_map = df.set_index("time_full")["cnt"]
prev_day_time = df["time_full"] - pd.Timedelta(days=1)
df["same_hour_prev"] = prev_day_time.map(same_hour_prev_map)

In [55]:
low, high = config.get_thresholds()
def assign_demand_level(cnt):
    if cnt <= low:
        return 0
    elif cnt <= high:
        return 1
    else: 
        return 2
df[config.TARGET_COLUMN] = df[config.TARGET_SOURCE_COLUMN].apply(assign_demand_level)
df[config.TARGET_COLUMN].map(config.DEMAND_LEVEL_LABELS).value_counts()

demand_level
high      7411
low       5158
medium    4810
Name: count, dtype: int64

In [56]:
for col, period in config.CYCLICAL_FEATURES.items():
    df[f"{col}_sin"] = np.sin(2 * np.pi * df[col]/period)
    df[f"{col}_cos"] = np.cos(2 * np.pi * df[col]/period)
df[[f"{c}_sin" for c in config.CYCLICAL_FEATURES] + [f"{c}_cos" for c in config.CYCLICAL_FEATURES]].head()

,hr_sin,mnth_sin,weekday_sin,hr_cos,mnth_cos,weekday_cos
0,0.000000,0.5,-0.781831,1.000000,0.866025,0.62349
1,0.258819,0.5,-0.781831,0.965926,0.866025,0.62349
2,0.500000,0.5,-0.781831,0.866025,0.866025,0.62349
3,0.707107,0.5,-0.781831,0.707107,0.866025,0.62349
4,0.866025,0.5,-0.781831,0.500000,0.866025,0.62349


In [57]:
df = pd.get_dummies(
    df, columns=config.ONE_HOT_FEATURES, prefix=config.ONE_HOT_FEATURES, drop_first=True
)
onehot_cols = [c for c in df.columns if c.startswith(tuple(f"{f}_" for f in config.ONE_HOT_FEATURES))]
df.head()

,instant,dteday,yr,mnth,hr,holiday,weekday,workingday,temp,atemp,hum,windspeed,casual,registered,cnt,time_full,gap_hours,lag_1h,same_hour_prev,demand_level,hr_sin,hr_cos,mnth_sin,mnth_cos,weekday_sin,weekday_cos,season_2,season_3,season_4,weathersit_2,weathersit_3,weathersit_4
0,1,2011-01-01,0,1,0,0,6,0,0.24,0.2879,0.81,0.0,3,13,16,2011-01-01 00:00:00,NaN,NaN,NaN,0,0.000000,1.000000,0.5,0.866025,-0.781831,0.62349,False,False,False,False,False,False
1,2,2011-01-01,0,1,1,0,6,0,0.22,0.2727,0.80,0.0,8,32,40,2011-01-01 01:00:00,1.0,16.0,NaN,0,0.258819,0.965926,0.5,0.866025,-0.781831,0.62349,False,False,False,False,False,False
2,3,2011-01-01,0,1,2,0,6,0,0.22,0.2727,0.80,0.0,5,27,32,2011-01-01 02:00:00,1.0,40.0,NaN,0,0.500000,0.866025,0.5,0.866025,-0.781831,0.62349,False,False,False,False,False,False
3,4,2011-01-01,0,1,3,0,6,0,0.24,0.2879,0.75,0.0,3,10,13,2011-01-01 03:00:00,1.0,32.0,NaN,0,0.707107,0.707107,0.5,0.866025,-0.781831,0.62349,False,False,False,False,False,False
4,5,2011-01-01,0,1,4,0,6,0,0.24,0.2879,0.75,0.0,0,1,1,2011-01-01 04:00:00,1.0,13.0,NaN,0,0.866025,0.500000,0.5,0.866025,-0.781831,0.62349,False,False,False,False,False,False



Theo nhận xét từ EDA (tương quan 0.992), bỏ `temp`, giữ `atemp`.

In [58]:
df = df.drop(columns=['temp'])

In [59]:
raw_time_cols = list(config.CYCLICAL_FEATURES.keys())
helper_cols = ["time_full", "gap_hours"] # bỏ cột phụ không dùng

df_sorted = df.drop(columns=raw_time_cols + helper_cols + config.DROP_COLUMNS)
df_sorted.columns.tolist()

['holiday',
 'workingday',
 'atemp',
 'hum',
 'windspeed',
 'lag_1h',
 'same_hour_prev',
 'demand_level',
 'hr_sin',
 'hr_cos',
 'mnth_sin',
 'mnth_cos',
 'weekday_sin',
 'weekday_cos',
 'season_2',
 'season_3',
 'season_4',
 'weathersit_2',
 'weathersit_3',
 'weathersit_4']

In [60]:
n = len(df)
n_test = int(n * config.TEST_SIZE)
n_val = int(n * config.VAL_SIZE)
n_train = n - n_test - n_val

train = df.iloc[:n_train].copy()
val = df.iloc[n_train:n_train+n_val].copy()
test = df.iloc[n_train + n_val:].copy()

print(f"train: {len(train)} lines")
print(f"val:   {len(val)} lines")
print(f"test:  {len(test)} lines")

train: 10429 lines
val:   3475 lines
test:  3475 lines


In [61]:
lag_cols = ["lag_1h", "same_hour_prev"]

final_feature_columns = [c for c in config.FEATURE_COLUMNS if c != "temp"] + onehot_cols + lag_cols
final_feature_columns


['hr_sin',
 'mnth_sin',
 'weekday_sin',
 'hr_cos',
 'mnth_cos',
 'weekday_cos',
 'holiday',
 'workingday',
 'atemp',
 'hum',
 'windspeed',
 'season_2',
 'season_3',
 'season_4',
 'weathersit_2',
 'weathersit_3',
 'weathersit_4',
 'lag_1h',
 'same_hour_prev']

## Chia tập train/val/test

In [62]:
x_train = train[final_feature_columns]
y_train = train[config.TARGET_COLUMN]

x_val = val[final_feature_columns]
y_val = val[config.TARGET_COLUMN]

x_test = test[final_feature_columns]
y_test = test[config.TARGET_COLUMN]

In [63]:
scaler = StandardScaler()
x_train_scale = scaler.fit_transform(x_train)
x_val_scale = scaler.transform(x_val)

## Feature Selection

Sử dụng 3 phương pháp, Mutual Information (model-free), Gini Importance, Permutation Importance để chọn các feature ảnh hưởng nhất tới điểm chính xác

In [64]:
#model_rf = RandomForestClassifier(random_state=seed, n_estimators=300)
#model_rf.fit(x_train_fit, y_train_fit)

In [65]:
mi_scores = mutual_info_classif(x_train, y_train, random_state=seed)
mi_series = pd.Series(mi_scores, index=final_feature_columns).sort_values(ascending=False)

# model_rf là mô hình Random Forest giả định đã được huẩn luyện trước 
# sau khi tiền xử lý, feature selection nhằm tìm ra các feature có độ ảnh hưởng
# cao để thử cải thiện mô hình 
# chạy feature selection trước khi hyperparameter tunnning
gini_series = pd.Series(model_rf.feature_importances_, index=final_feature_columns).sort_values(ascending=False)

perm_result = permutation_importance(
    model_rf, x_val, y_val, n_repeats=10, random_state=seed, scoring="f1_macro"
)
perm_series = pd.Series(perm_result.importances_mean, index=final_feature_columns).sort_values(ascending=False)

importance_df = pd.DataFrame({
    "mutual_info": mi_series,
    "gini": gini_series,
    "permutation": perm_series,
})
importance_df

ValueError: Input X contains NaN.

In [ ]:
normalized = importance_df.apply(lambda col: col / col.max())
normalized["combined_score"] = normalized.mean(axis=1)
normalized = normalized.sort_values("combined_score", ascending=False)

fig, ax = plt.subplots(figsize=(8, 6))
sns.barplot(x=normalized["combined_score"], y=normalized.index, ax=ax)
ax.set_title("Combined feature importance (MI + Gini + Permutation, normalized)")
plt.tight_layout()
plt.show()

normalized

## Chọn tập feature rút gọn

In [ ]:
median_score = normalized["combined_score"].median()
selected_features = normalized[normalized["combined_score"] >= median_score].index.tolist()

print("So luong feature con lai:", len(selected_features), "/", len(final_feature_columns))
print(selected_features)